# `POST /validate-config` — assignment examples

Manual checks against the running server. Each cell sends input/output examples (also from `specs/assignment.md`) to `POST /validate-config` and prints the prettified JSON response.

**Prereqs**

- Server running locally: `npm run dev:server` (or `npm run dev`).
- `OPENAI_API_KEY` set in `server/.env` (otherwise the endpoint returns `502`).
- Python `requests` available: `pip install requests`.

In [ ]:
import json
import requests

BASE_URL = "http://localhost:3000"
ENDPOINT = f"{BASE_URL}/validate-config"
MODEL = "gpt-5"  # one of "gpt-4o-mini", "gpt-4o" or "gpt-5"

def call_validate(config: dict, model: str | None = MODEL) -> None:
    """POST `config` to /validate-config and pretty-print the response."""
    params = {"model": model} if model else None
    print("Request:")
    print(json.dumps(config, indent=2))
    print()
    response = requests.post(ENDPOINT, json=config, params=params, timeout=120)
    print(f"HTTP {response.status_code}")
    try:
        body = response.json()
        print(json.dumps(body, indent=2, ensure_ascii=False))
    except ValueError:
        print(response.text)

### The results are according to the following reference ranges
```json
{
  "difficulties": {
    "easy": {
      "reward_min": 100,
      "reward_max": 500,
      "time_limit_min": 30
    },
    "medium": {
      "reward_min": 500,
      "reward_max": 2000,
      "time_limit_min": 20,
      "time_limit_max": 60
    },
    "hard": {
      "reward_min": 2000,
      "reward_max": 5000,
      "time_limit_min": 10,
      "time_limit_max": 30
    }
  },
  "total_levels": 150
}

```

## Example 1 — reward too high for an easy level

Expected pattern: schema is valid; the LLM should flag a `reward_vs_difficulty` mismatch (5000 reward on `easy`).

In [2]:
call_validate({
    "level": 12,
    "time_limit": 60,
    "reward": 5000,
    "difficulty": "easy"
})

Request:
{
  "level": 12,
  "time_limit": 60,
  "reward": 5000,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Early level 12 is marked easy with a generous 60s timer (valid for easy), but it grants a 5000 reward, which sits at the top of the hard tier and far exceeds easy’s reward band. Progression position aligns with easy difficulty; the mismatch is purely reward vs. declared difficulty.",
    "suggested_actions": [
      "Reduce reward to 100–500 for easy difficulty, or reclassify the level as hard."
    ],
    "confidence": 0.98
  }
}


## Example 2 — time limit too tight for a hard level

Expected pattern: schema is valid; the LLM should flag `time_vs_difficulty` (10s on `hard`) and likely `frustration_risk`.

In [3]:
call_validate({
    "level": 5,
    "time_limit": 10,
    "reward": 500,
    "difficulty": "hard"
})

Request:
{
  "level": 5,
  "time_limit": 10,
  "reward": 500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 5 is labeled hard with a 10s timer (right on the hard minimum) but only 500 reward, which aligns with easy/medium payouts. The level sits very early in progression, creating a mismatch between declared difficulty, reward magnitude, and position.",
    "suggested_actions": [
      "Increase reward to 2000–5000 to match hard difficulty.",
      "Move this hard level to a late-game position or downgrade its difficulty to fit early progression."
    ],
    "confidence": 0.93
  }
}


## Example 3 — reasonable starting level (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array, so `suggested_actions` is `["No action needed"]` and `confidence` is the model's `verdict_confidence`.

In [4]:
call_validate({
    "level": 1,
    "time_limit": 120,
    "reward": 100,
    "difficulty": "easy"
})

Request:
{
  "level": 1,
  "time_limit": 120,
  "reward": 100,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is marked easy with the minimum reward for that tier and a very generous time limit. Both reward and timer sit within the easy ranges, and the early position aligns with an easy difficulty.",
    "suggested_actions": [
      "No action needed"
    ],
    "confidence": 0.96
  }
}


## Example 4 — early level marked as hard (level_vs_difficulty)

Expected pattern: schema is valid; the LLM should flag `level_vs_difficulty` because level 1 of 150 is the very start of the progression while the declared difficulty is `hard`. `reward` (3000) and `time_limit` (20s) both stay inside the hard tier, so the field-vs-difficulty rules should not fire.

In [5]:
call_validate({
    "level": 1,
    "time_limit": 20,
    "reward": 3000,
    "difficulty": "hard"
})

Request:
{
  "level": 1,
  "time_limit": 20,
  "reward": 3000,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is configured as hard with a 20s timer and 3000 reward, both well within the hard ranges. However, placing a hard level at the very start of a 150-level progression is a strong mismatch for expected early-game difficulty.",
    "suggested_actions": [
      "Lower this level to easy/medium or move it far later in the progression."
    ],
    "confidence": 0.95
  }
}


## Example 5 — runaway reward (economy_risk)

Expected pattern: schema is valid; a 50000 reward is roughly 10× the hard tier's max (5000) and would distort the long-term currency economy if repeated across many levels — the LLM should flag `economy_risk`. `reward_vs_difficulty` will likely fire too, since the value is far above the declared hard range.

In [6]:
call_validate({
    "level": 140,
    "time_limit": 25,
    "reward": 50000,
    "difficulty": "hard"
})

Request:
{
  "level": 140,
  "time_limit": 25,
  "reward": 50000,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Late-game hard level (140/150) with a reasonable hard-tier timer (25s within 10–30s), but the reward is massively above the hard range. This creates a severe reward-to-difficulty mismatch and a strong risk of economy inflation if repeated.",
    "suggested_actions": [
      "Reduce reward to 2000–5000 for hard difficulty.",
      "Cap the reward at or below 5000 to prevent currency inflation in the main progression."
    ],
    "confidence": 0.99
  }
}


## Example 6 — tight timer with low-end reward inside medium tier (frustration_risk)

Expected pattern: schema is valid; each field individually sits inside the medium tier (`reward` near the floor at 550, `time_limit` near the floor at 22s), so neither field-vs-difficulty rule fires on its own. The combination — tight timer paired with a low-end reward — is the "grind to retry" pattern named in the system prompt, and the LLM should flag `frustration_risk`.

In [7]:
call_validate({
    "level": 75,
    "time_limit": 22,
    "reward": 550,
    "difficulty": "medium"
})

Request:
{
  "level": 75,
  "time_limit": 22,
  "reward": 550,
  "difficulty": "medium"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 75 (mid-game) is marked medium with a 22s timer and a 550 reward, both near the low end of the medium ranges. This creates a high-pressure, low-payout profile only slightly above easy rewards.",
    "suggested_actions": [
      "Increase reward to around 800–1200 or extend the time_limit to 30–40s to better align effort with payout."
    ],
    "confidence": 0.68
  }
}


## Example 7 — multiple rules violated at once

Expected pattern: schema is valid; this configuration breaks several rules — `level_vs_difficulty` (level 1 declared `hard`), `reward_vs_difficulty` (200 sits in the easy band, not hard), and `time_vs_difficulty` (5s is below the hard tier's minimum of 10s). Expect multiple findings in a single response.

In [8]:
call_validate({
    "level": 1,
    "time_limit": 5,
    "reward": 200,
    "difficulty": "hard"
})

Request:
{
  "level": 1,
  "time_limit": 5,
  "reward": 200,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is labeled hard but offers an easy-tier reward and an extremely tight 5s timer below hard’s minimum. Its early position also contradicts a hard difficulty placement. Overall, the level mixes early-game placement with late-game labeling and mismatched parameters.",
    "suggested_actions": [
      "Increase reward to 2000–5000 for hard difficulty.",
      "Raise the time_limit to at least 10 seconds (ideally 10–30s) to fit hard difficulty.",
      "Set this level’s difficulty to easy given its position as level 1."
    ],
    "confidence": 0.95
  }
}


## Example 8 — late-game hard, in tier (expect empty findings)

Expected pattern: schema is valid; level 150 of 150 is the very end of the progression so `hard` fits, and `reward` / `time_limit` both sit comfortably inside the hard tier. The LLM should return an empty `findings` array, so `suggested_actions` is `["No action needed"]` and `confidence` is the model's `verdict_confidence`.

In [9]:
call_validate({
    "level": 150,
    "time_limit": 25,
    "reward": 4500,
    "difficulty": "hard"
})

Request:
{
  "level": 150,
  "time_limit": 25,
  "reward": 4500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Final level (150/150) marked hard with a 25s timer and 4500 reward; both sit comfortably within the hard ranges. The late-game position matches a hard difficulty expectation. Values are near the upper ends but remain compliant with defined bounds.",
    "suggested_actions": [
      "No action needed"
    ],
    "confidence": 0.95
  }
}


## Example 9 — schema-validation failure (expect HTTP 400, no LLM call)

Sends a malformed body to confirm the Zod gate rejects it before the LLM is called.

In [10]:
call_validate({"level": "oops"})

Request:
{
  "level": "oops"
}

HTTP 400
{
  "schema_validation": {
    "valid": false,
    "errors": [
      {
        "path": "level",
        "message": "Expected number, received string"
      },
      {
        "path": "time_limit",
        "message": "Required"
      },
      {
        "path": "reward",
        "message": "Required"
      },
      {
        "path": "difficulty",
        "message": "Required"
      }
    ]
  }
}
